Groq api key generation : https://console.groq.com/keys





In [ ]:
#CELL 1
!pip install groq --quiet

import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print('Libraries imported successfully')

Libraries imported successfully


In [ ]:
from groq import Groq
API_KEY = "xxxxxxxxxxxxxx"
client = Groq(api_key=API_KEY)
MODEL = "llama-3.1-8b-instant"

In [ ]:
print(f"Groq client configured with model : {MODEL}")
print("Make sure the api key is replaced with actual key")

Groq client configured with model : llama-3.1-8b-instant
Make sure the api key is replaced with actual key


In [ ]:
def ask_llm(username, system_message="You are the helpful assistent.",temperature=0.7, max_tokens=500):
  response =client.chat.completions.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": system_message},
          {"role": "user", "content": f"{username}"},
      ],
      temperature=temperature,
      max_tokens=max_tokens,
  )
  return response.choices[0].message.content

In [ ]:
test_response=ask_llm("What is ETL? in short 2 sentence.")
test_response2=ask_llm("What is capital of India just one line")
print("==LLM RESPONSE==")
print(test_response)
print(test_response2)

==LLM RESPONSE==
ETL (Extract, Transform, Load) is a process used to extract data from various sources, transform it into a standardized format, and load it into a target system, such as a data warehouse or a database. This process helps to organize and analyze large amounts of data from different sources, making it easier to gain insights and make informed decisions.
The capital of India is New Delhi.


In [ ]:
response_etl = ask_llm(
    "In a 3 bullet points, explain how the Medallion Architecture "
    "(Bronze, Silver, Gold layer) related to ETL pipelines.",
    system_message="you are a senior data engineering instructor."
                   "Be concise and practical."
)
print('Medallion + ETL connection:')
print(response_etl)
print()
print("--- Token explanation ---")
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately', len(response_etl.split())*1.3, 'tokens.')
print('Llama-3.1-8b context window: 8192 tokens (~6000 words per conversation)')

Medallion + ETL connection:
As a senior data engineering instructor, I can explain the Medallion Architecture in relation to ETL pipelines as follows:

• **Bronze Layer (Raw Data Ingestion)**: This layer is responsible for ingesting raw data from various sources, such as logs, APIs, or databases. The Bronze layer is often the entry point for ETL pipelines, where data is collected, processed, and stored in a raw, untransformed format, usually in a data lake or a NoSQL database.

• **Silver Layer (Data Processing and Transformation)**: After raw data is ingested, it's processed and transformed in the Silver layer. This is where data is cleaned, aggregated, and transformed into a standardized format, often in a data warehouse or a relational database. The Silver layer is the heart of the ETL pipeline, where complex data processing and transformations occur.

• **Gold Layer (Data Marts and Business Intelligence)**: In the Gold layer, transformed data is stored in optimized data marts for b

In [ ]:
zero_shot_response = ask_llm(
    "Extract the city name from this address: "
    "456 Briage Road , Bangalore 560025, Karnataka, india"
)
print('Result of the zero shot: ')
print(zero_shot_response)
print()

Result of the zero shot: 
The city name in the given address is "Bangalore".



In [ ]:
ambiguous_response=ask_llm("Clean this data : ramesh Kumar, 45000,mumbai")
print('Ambiguous zero shot result :')
print(ambiguous_response)
print()
print("problem : output format is unpredictable and not machine-parseable!")

Ambiguous zero shot result :
The data appears to be in an inconsistent format, but I'll try to clean it up. Based on the provided data, "ramesh Kumar, 45000, mumbai", here's a cleaned version:

1. First name: Ramesh
2. Last name: Kumar
3. Salary: 45000
4. Location: Mumbai

If you provided multiple data points, I'd be happy to help clean them up as well.

problem : output format is unpredictable and not machine-parseable!


In [ ]:
few_shot_prompt="""
Convert the employee text to json. Here is the examples:
Input : ramesh kumar, 45000, mumbai
Output : {"name": "ramesh kumar", "salary": "45000", "city":"mumbai"}
Input : priya nair, 52000, Delhi
Output : {"name": "priya nair", "salary": "52000", "city":"Delhi"}
Now Conver this:
Input :ANANYA DAS, 38000, Kolkata
Output:"""

few_shot_response=ask_llm(few_shot_prompt,temperature=0.0)
print('Few shot result :')
print(few_shot_response)
print()

try:
  parsed=json.loads(few_shot_response.strip())
  print("Succesfully parsed as JSON!")
  print(f"Name: {parsed["name"]}, Salary : {parsed["salary"]}, City : {parsed["city"]}")
except json.JSONDecodeError:
  print("Parsing failed -model added extra text")
  print("Solution: add the explicit instructions in the system prompt")


Few shot result :
To convert the employee text to JSON, we can use the following Python code:

```python
import json

def convert_to_json(name, salary, city):
    employee = {
        "name": name,
        "salary": str(salary),
        "city": city
    }
    return json.dumps(employee)

# Test the function
name = "ANANYA DAS"
salary = 38000
city = "Kolkata"
print(convert_to_json(name, salary, city))
```

When you run this code, it will output:

```json
{"name": "ANANYA DAS", "salary": "38000", "city": "Kolkata"}
```

This code defines a function `convert_to_json` that takes three parameters: `name`, `salary`, and `city`. It creates a dictionary `employee` with these parameters and then uses the `json.dumps` function to convert the dictionary to a JSON string. The `str(salary)` conversion is necessary because JSON does not support direct conversion of integers to strings.

Parsing failed -model added extra text
Solution: add the explicit instructions in the system prompt


role prompting

In [ ]:
same_question = "Review this Python code and identity any issues:\n" \
"df['revenue']=df['qty']* df['price']\n" \
"result=df.groupby('dept').sum()"

generic_response=ask_llm(same_question, temperature=0.2)
print('Without Role Prompting')
print(generic_response[:300], '...')
print()

Without Role Prompting
The provided Python code appears to be a simple data manipulation task using the pandas library. However, there are a few potential issues that can be identified:

1. **Missing Error Handling**: The code does not include any error handling. If the 'qty' or 'price' columns do not exist in the DataFra ...



In [ ]:
prompt="Give me one creative name for the data analytics startup"
print("===Temperature===")
for temp in[0.0,0.5,1.0]:
  print(f"Temperature : {temp}")
  response=ask_llm(prompt,temperature=temp)
  print(response.strip())

  time.sleep(1)

  print()
  print('Observation:')
  print('Temperature=0.0 -. same or very similar answer every run (deterministic)')
  print('Temperature=0.5 -> some variation')
  print("Temperature=1.0 -> more creative/varied, sometimes surprising")
  print()
  print('Rule for data engineering tasks: use temperature =0.0 or 0.1')
  print('You need the consistent, parseable output - not creative variation')

===Temperature===
Temperature : 0.0
Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect data points and provide insights that were previously unknown. This name also has a modern and tech-savvy feel to it, which is fitting for a data analytics startup.
Temperature : 0.5
Here's a creative name for a data analytics startup:

**"Nexa Insights"**

"Nexa" suggests connection and linkages, implying that the startup helps connect data points to provide valuable insights. This name also has a modern and tech-savvy feel to it, which is fitting for a data analytics startup.
Temperature : 1.0
One creative name for a data analytics startup could be: **Numina Insights**

"Numina" is derived from the Latin word for "spirit" or "energy," which can be interpreted as the invisible forces that drive data and help businesses make informed decisions. Adding "insights" to the name conveys the startup's focus on pr

In [ ]:
invoice_text = "Invoice #2024-001 from TECHWORLD SOLUTIONS dated 15th January 2024. Amount: Rs. 45,000 for laptop"

weak_response=ask_llm(
    f"Clean this invoice data:{invoice_text}",
    temperature=0.3
)
print('WEAK PROMPT OUTPUT: ')
print(weak_response)
print()

try:
  json.loads(weak_response)
  print('PARSEABLE: Yes')
except:
  print('PARSEABLE: No - cannot load into DataFrame')
print('\n'+'='*50 +'\n')

WEAK PROMPT OUTPUT: 
Here's the cleaned invoice data:

**Invoice Details:**

- **Invoice Number:** 2024-001
- **Invoice Date:** 15th January 2024
- **Invoice Issuer:** TECHWORLD SOLUTIONS
- **Invoice Description:** Laptop
- **Invoice Amount:** Rs. 45,000

I cleaned the data by:

- Breaking down the date into day, month, and year for clarity
- Adding the issuer's name for identification
- Adding the description for clarity
- Keeping the amount in the original currency for consistency



In [ ]:
#strong prompt
strong_system="""You are a data extraction specialist for an accounting pipeline.
Extract invoive data and return only a valid JSON object.
Do not include any explanation, preamble, or markdown formatting.
Return only the JSON, noting else.

JSON schema (use null for missing values):
{"invoice id" : string , "vendor_name" :string (title case),
"amount" : number (no currency symbols),
"currency" : string (default INR),
"invoice_date" : string (YYYY-MM-DD),
"category" : string (Electronics/Services/Accessories/Others)
}
"""
strong_response=ask_llm(
    f'Extract from :{invoice_text}',
    system_message=strong_system,
    temperature=0.0
)
print('STRONG PROMPT OUTPUT: ')
print(strong_response)
print()

try:
  parsed =json.loads(strong_response.strip())
  print('PARSEABLE : Yes')
  print(f'Vendor:{parsed.get("vendor_name")}')
  print(f'Amount:{parsed.get("amount")}')
  print(f'Date:{parsed.get("invoice_date")}')
except json.JSONDecodeError:
  match = re.search(r'\{.?\}', strong_response, re.DOTALL)
  if match:
    parsed = json.loads(match.group())
    print('Parsable : Yes (extracted with regex fall back)')
  else:
    print('Parsable : no - retry with stricket prompt')


STRONG PROMPT OUTPUT: 
{"invoice id": "2024-001", "vendor_name": "Techworld Solutions", "amount": 45000, "currency": "INR", "invoice_date": "2024-01-15", "category": "Electronics"}

PARSEABLE : Yes
Vendor:Techworld Solutions
Amount:45000
Date:2024-01-15


MINI PROJECT: Smart Data Cleaner

Goal: Convert 5 messy invoice strings into a clean, structured Pandas Dataframe using LLM

This is a complete GenAI-powered ETL pipeline.

Messy Text-> LLM -> JSON -> DataFrame ->Analysis

Q1. What is the difference between ML (day 5) and Generative AI(Day 6)

Q2. What does temperature=0.0 do in an LLM API call and when would you use it?

Q3. Write a few-shot prompt that extracts name and salary from text in JSON format.

Q4. What is LLM hallucination and how can prompt engineering reduce it?

Q5. Your LLM returns ```json\n{"name":"Ramesh"}\n and json.loads() crashes. Write the fix.

*Q6:*How does today's

### Short Answers to Your Questions:

#### Q1. Difference between ML and Generative AI
**ML** focuses on learning patterns from data to make predictions or decisions. **Generative AI** is a type of ML that creates new, original content (like text or images) based on learned patterns.

#### Q2. What does `temperature=0.0` do?
`temperature=0.0` makes the LLM's output highly deterministic and consistent (least creative). Use it for tasks requiring precise, repeatable results, such as data extraction and parsing.

#### Q3. Few-shot prompt for name and salary extraction in JSON

In [ ]:
few_shot_prompt_short = '''
Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:
'''
print('Short Few-Shot Prompt:')
print(few_shot_prompt_short)
# Expected output for 'ALICE JOHNSON, 82000' would be: {"name": "Alice Johnson", "salary": 82000}

Short Few-Shot Prompt:

Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:



#### Q4. LLM hallucination and how prompt engineering reduces it
**LLM hallucination** is when the model generates false or nonsensical information. **Prompt engineering** reduces it by providing clear instructions, context, examples, and strict output constraints (e.g., "Answer only from provided text", "Return ONLY JSON").

#### Q5. Fix for `json.loads()` crashing on ```json\n{"name":"Ramesh"}\n

In [ ]:
import json
import re

llm_output_bad = '```json\n{"name":"Ramesh"}\n```'

# Use regex to extract the pure JSON string
match = re.search(r'```json\n(.*)\n```', llm_output_bad, re.DOTALL)
if match:
    json_string_fixed = match.group(1).strip()
    try:
        parsed_data = json.loads(json_string_fixed)
        print(f"Fixed: Successfully parsed as: {parsed_data}")
    except json.JSONDecodeError as e:
        print(f"Error even after regex: {e}")
else:
    print("No JSON block found.")

Fixed: Successfully parsed as: {'name': 'Ramesh'}


#### Q6. Smart Cleaner vs. Manual ETL Cleaning
Today's **Smart Cleaner** (LLM-based) automatically extracts and structures data from *unstructured text* using natural language understanding, adapting flexibly to variations. **Manual ETL** relies on predefined rules for *structured/semi-structured data* and requires more effort to adapt to new formats.

In [ ]:
# Make sure to run all preceding cells, especially the one defining `ask_llm`.
few_shot_prompt = """
Convert employee text to JSON. Here are examples:

Input: RAMESH KUMAR, 45000, mumbai
Output: {"name": "Ramesh Kumar", "salary": 45000, "city": "Mumbai"}

Input: priya nair, 52000, Delhi
Output: {"name": "Priya Nair", "salary": 52000, "city": "Delhi"}

Now convert this:
Input: ANANYA DAS, 38000, kolkata
Output:
"""
few_shot_response = ask_llm(few_shot_prompt, temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()

try:
  # The type hint 'parsed: json.loads' is incorrect. It should be 'parsed = json.loads'
  parsed = json.loads(few_shot_response.strip())
  print('Sucessfully parsed as JSON!')
  print(f'Name: {parsed["name"]}, Salary: {parsed["salary"]}, City: {parsed["city"]}')

except json.JSONDecodeError:
  print('Parsing failed - model added extra text or invalid JSON.')
  print("Solution: add explicit instructions in the system prompt to return ONLY JSON, and check your prompt examples for correct JSON syntax.")


Few-Shot Result:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(', ')
    
    # Create a dictionary with the given keys
    employee_data = {
        "name": values[0].strip().title(),
        "salary": int(values[1]),
        "city": values[2].strip().title()
    }
    
    # Convert the dictionary to JSON
    json_data = json.dumps(employee_data)
    
    return json_data

# Test the function
employee_text = "ANANYA DAS, 38000, kolkata"
print(convert_to_json(employee_text))
```

When you run this function with the input "ANANYA DAS, 38000, kolkata", it will output:

```json
{"name": "Ananya Das", "salary": 38000, "city": "Kolkata"}
```

This function works by splitting the input string into individual values using the `split(', ')` method. It then creates a dictionary with the given keys and assigns the corresponding v